# **Trabajo Fin de Máster de Patricia Olguín**
## Capítulo comparativo: Argentina, Brasil y España

Este notebook complementa al principal (`01_TFM_salarios_mujeres_Argentina.ipynb`). La pregunta es: **¿premian los
mercados de trabajo de Argentina, Brasil y España los mismos factores, y en la misma medida?**

Para comparar con rigor se usa **solo un conjunto de variables comunes** a las tres encuestas y la misma variable
objetivo relativa (salario horario sobre la mediana del país y período). Se comparan **estructuras** (cuánto pesa cada
factor), no niveles salariales en monedas distintas.

| País | Encuesta | Organismo | Ondas | Salario |
|---|---|---|---|---|
| Argentina | Encuesta Permanente de Hogares | INDEC | 1T2024 – 1T2026 | Neto, mes anterior |
| Brasil | PNAD Contínua | IBGE | 1T2024 – 2T2026 | Bruto habitual |
| España | Encuesta de Condiciones de Vida (EU-SILC) | INE | ECV 2024 y 2025 | Neto anual del año anterior |
| España (validación) | Encuesta de Estructura Salarial | INE | 2022 | Bruto de nómina |

**Requisito:** haber ejecutado antes el notebook principal, que deja la base argentina procesada en Drive.

## 1. Configuración del entorno

In [ ]:
import io
import os
import re
import subprocess
import sys
import zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "lightgbm"], check=False)

import duckdb
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
import seaborn as sns
import statsmodels.formula.api as smf
from IPython.display import Markdown, display
from sklearn.metrics import r2_score

# Mediana ponderada (la misma función de funciones_tfm.py). Se define aquí para que este notebook no dependa de
# que el repositorio esté clonado en la sesión: cada notebook de Colab corre en su propia máquina.
def mediana_ponderada(valores, pesos):
    valores, pesos = np.asarray(valores, float), np.asarray(pesos, float)
    orden = np.argsort(valores)
    acumulado = np.cumsum(pesos[orden])
    return valores[orden][np.searchsorted(acumulado, 0.5 * acumulado[-1])]


import warnings
warnings.filterwarnings("ignore")
sns.set()
SEMILLA = 42

ruta_drive = os.environ.get("TFM_DRIVE", "/content/drive/MyDrive")
RAW = Path(ruta_drive) / "TFM" / "datos_comparacion"          # microdatos de Brasil y España
PROC = Path(ruta_drive) / "TFM" / "datos_procesados"
for carpeta in ["pnadc", "ecv", "ees"]:
    (RAW / carpeta).mkdir(parents=True, exist_ok=True)

ONDAS_BR = [(a, q) for a in (2024, 2025) for q in (1, 2, 3, 4)] + [(2026, 1), (2026, 2)]
ANIOS_ECV = [2024, 2025]

## 2. Descarga de los microdatos de Brasil y España

Los archivos se descargan una sola vez de los servidores oficiales del IBGE y del INE y quedan guardados en Drive.
La PNAD Contínua pesa unos 220 MB por trimestre, así que la primera descarga puede tardar varios minutos.

In [ ]:
IBGE = "https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados"
INE_ECV = "https://www.ine.es/dyngs/INEbase/es/operacion.htm?c=Estadistica_C&cid=1254736176807&menu=resultados&secc=1254736195153&idp=1254735976608"


def url_ibge(carpeta, patron):
    """Busca el archivo en el listado de la carpeta del IBGE. El nombre puede llevar la fecha de publicación
    (por ejemplo PNADC_012024_20250815.zip) o no (PNADC_012026.zip): se elige la versión más reciente."""
    listado = requests.get(f"{IBGE}/{carpeta}/", timeout=60).text
    nombres = sorted(set(re.findall(patron, listado)))
    if not nombres:
        raise FileNotFoundError(f"No se encontró {patron} en {IBGE}/{carpeta}/")
    return f"{IBGE}/{carpeta}/{nombres[-1]}"


# Archivo local -> función que devuelve la dirección de descarga (se consulta solo si el archivo falta)
DESCARGAS = {RAW / "pnadc" / "Dicionario_e_input.zip": lambda: url_ibge("Documentacao", r"Dicionario_e_input_\d+\.zip")}
DESCARGAS.update({RAW / "pnadc" / f"PNADC_{q:02d}{a}.zip":
                  (lambda a=a, q=q: url_ibge(a, rf"PNADC_{q:02d}{a}(?:_\d{{8}})?\.zip")) for a, q in ONDAS_BR})
DESCARGAS.update({RAW / "ecv" / f"datos_{a}.zip": (lambda a=a: f"https://www.ine.es/ftp/microdatos/ecv/ecv_b2013/datos_{a}.zip")
                  for a in ANIOS_ECV})
DESCARGAS[RAW / "ees" / "datos_2022.zip"] = lambda: "https://www.ine.es/ftp/microdatos/salarial/datos_2022.zip"

for destino, url in DESCARGAS.items():
    if destino.exists():
        continue
    try:
        direccion = url()
        print("Descargando", direccion)
        with requests.get(direccion, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(destino, "wb") as f:
                for bloque in r.iter_content(1 << 20):
                    f.write(bloque)
    except Exception as e:
        # Si un servidor cambia el nombre de un archivo, se puede descargar a mano y guardar con el nombre indicado
        destino.unlink(missing_ok=True)
        print(f"No se pudo descargar {destino.name}: {e}")
        print(f"   Descárguelo a mano ({'IBGE: ' + IBGE if 'pnadc' in str(destino) else 'INE: ' + INE_ECV}) "
              f"y guárdelo en Drive como {destino}")
faltan = [d.name for d in DESCARGAS if not d.exists()]
print("Microdatos disponibles:", len(DESCARGAS) - len(faltan), "de", len(DESCARGAS), "| faltan:", faltan or "ninguno")

## 3. Variables comunes y clasificaciones compartidas

| Variable común | Argentina (EPH) | Brasil (PNADC) | España (ECV) |
|---|---|---|---|
| Salario mensual | `P21` neto | `VD4016` bruto habitual | `PY010N`/12 neto |
| Horas semanales | `PP3E_TOT` | `V4039` | `PL060` |
| Ponderador | `PONDIIO` | `V1028` | `PB040` |
| Años de educación | `CH12`-`CH14` | `VD3005` | `PE041` (CINE → años) |
| Calificación (3 niveles) | CNO-2001 | COD (≈ CIUO-08) | CIUO-08 |
| Rama | CAES-Mercosur | CNAE-Domiciliar 2.0 | CNAE-2009 |
| Empleo registrado | `PP07H` | `VD4012` | `PL141` contrato escrito |
| Contrato temporal | `PP07C` | `V4025` | `PL141` duración determinada |
| Sector público | `PP04A` | `VD4009` 05-07 | — |
| Jefa de hogar / en pareja | `CH03` / `CH07` | `V2005` | — / `PB205` |
| Menores de 6 y 18, otros ocupados | base de personas | base de personas | fichero R |
| Gran ciudad | aglomerado > 500 mil | capital o región metropolitana | zona densamente poblada |

La calificación se agrega en tres niveles comparables: **alta** (directivos, profesionales y técnicos), **media**
(administrativos, servicios, oficios, operadores) y **baja** (ocupaciones elementales).

In [ ]:
SECCION_A_RAMA = {"A": "Agro y minería", "B": "Agro y minería", "C": "Industria", "D": "Construcción y servicios básicos",
                  "E": "Construcción y servicios básicos", "F": "Construcción y servicios básicos", "G": "Comercio",
                  "H": "Transporte e información", "J": "Transporte e información", "I": "Hoteles y restaurantes",
                  "K": "Finanzas y servicios a empresas", "L": "Finanzas y servicios a empresas",
                  "M": "Finanzas y servicios a empresas", "N": "Finanzas y servicios a empresas",
                  "O": "Administración pública", "P": "Enseñanza", "Q": "Salud y servicios sociales", "R": "Otros servicios",
                  "S": "Otros servicios", "U": "Otros servicios", "T": "Servicio doméstico"}
DIVISION_A_SECCION = [((1, 3), "A"), ((5, 9), "B"), ((10, 33), "C"), ((35, 35), "D"), ((36, 39), "E"), ((40, 43), "F"),
                      ((45, 48), "G"), ((49, 53), "H"), ((55, 56), "I"), ((58, 63), "J"), ((64, 66), "K"), ((68, 68), "L"),
                      ((69, 75), "M"), ((77, 82), "N"), ((83, 84), "O"), ((85, 85), "P"), ((86, 88), "Q"), ((90, 93), "R"),
                      ((94, 96), "S"), ((97, 98), "T"), ((99, 99), "U")]


def rama_comun(division):
    if pd.isna(division):
        return np.nan
    for (lo, hi), seccion in DIVISION_A_SECCION:
        if lo <= int(division) <= hi:
            return SECCION_A_RAMA[seccion]
    return np.nan


def calificacion_isco(gran_grupo):
    if pd.isna(gran_grupo):
        return np.nan
    return "alta" if gran_grupo in (1, 2, 3) else ("baja" if gran_grupo == 9 else "media")


COMUNES = ["anios_educ", "exp_pot", "exp_pot2", "calificacion", "rama", "formal", "temporal", "jornada_parcial",
           "en_pareja", "menores_6", "menores_18", "otros_ocupados", "gran_ciudad"]

## 4. Argentina: base procesada del notebook principal

In [ ]:
ar = pd.read_parquet(PROC / "base_procesada.parquet")
AR = pd.DataFrame({
    "pais": "Argentina", "periodo": ar["periodo"].astype(str), "t": ar["t"], "id_hogar": ar["id_hogar"],
    "peso": ar["PONDIIO"], "mujer": (ar["CH04"] == 2).astype(int), "salario_horario": ar["salario_horario"],
    "anios_educ": ar["anios_educ"], "exp_pot": ar["exp_pot"],
    "calificacion": ar["calificacion"].map({"Profesional": "alta", "Técnica": "alta", "Operativa": "media",
                                            "No calificada": "baja"}),
    "rama": ar["rama"].replace({"Agro y pesca": "Agro y minería", "Minería": "Agro y minería",
                                "Electricidad, gas y agua": "Construcción y servicios básicos",
                                "Construcción": "Construcción y servicios básicos", "Transporte": "Transporte e información",
                                "Información y comunicación": "Transporte e información",
                                "Finanzas e inmobiliarias": "Finanzas y servicios a empresas",
                                "Servicios profesionales y empresariales": "Finanzas y servicios a empresas",
                                "Organismos extraterritoriales": "Otros servicios"}),
    "formal": ar["PP07H"], "temporal": ar.get("PP07C", pd.Series(np.nan, index=ar.index)),
    "jornada_parcial": ar["jornada_parcial"], "en_pareja": ar["estado_civil"].isin(["Unida", "Casada"]).astype(float),
    "menores_6": ar["menores_6"], "menores_18": ar["menores_18"], "otros_ocupados": ar["otros_ocupados_hogar"],
    "gran_ciudad": (ar["MAS_500"].astype(str).str.upper() == "S").astype(float)})
print(f"Argentina: {len(AR):,} asalariados ({AR.mujer.sum():,} mujeres)")

## 5. Brasil: PNAD Contínua

Los microdatos son archivos de **ancho fijo** (cada variable ocupa posiciones fijas de cada línea) de cientos de MB por
trimestre: más de 4 millones de registros en el período. Se procesan con **DuckDB**, un motor SQL que lee el texto sin
cargarlo entero en memoria. Las posiciones de cada variable se toman del programa de lectura que publica el IBGE.

In [ ]:
COLS_PNADC = ["Ano", "Trimestre", "UF", "UPA", "V1008", "V1014", "V1023", "V1028", "V2003", "V2005", "V2007", "V2009",
              "VD3005", "VD4002", "VD4009", "VD4012", "VD4016", "V4039", "V4010", "V4013", "V4025"]
TEXTO = {"UPA", "V1008", "V1014", "V2003", "V4010", "V4013"}

with zipfile.ZipFile(RAW / "pnadc" / "Dicionario_e_input.zip") as z:
    programa = z.read(next(n for n in z.namelist() if re.search(r"input.*\.txt$", n, re.I))).decode("latin-1")
posiciones = {nombre: (int(inicio), int(ancho)) for inicio, nombre, ancho in re.findall(r"@(\d+)\s+(\w+)\s+\$?(\d+)\.", programa)}

parquets = []
for a, q in ONDAS_BR:
    destino = PROC / "pnadc" / f"pnadc_{a}_{q}.parquet"
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists():
        with zipfile.ZipFile(RAW / "pnadc" / f"PNADC_{q:02d}{a}.zip") as z:
            txt = Path(z.extract(next(n for n in z.namelist() if n.lower().endswith(".txt")), "/tmp"))
        columnas = ", ".join(
            f"trim(substr(line, {posiciones[c][0]}, {posiciones[c][1]})) AS {c}" if c in TEXTO else
            f"TRY_CAST(NULLIF(trim(substr(line, {posiciones[c][0]}, {posiciones[c][1]})), '') AS DOUBLE) AS {c}"
            for c in COLS_PNADC)
        duckdb.sql(f"COPY (SELECT {columnas} FROM read_csv('{txt}', columns={{'line': 'VARCHAR'}}, header=false, "
                   f"delim='\\t', quote='', escape='', auto_detect=false)) TO '{destino}' (FORMAT PARQUET)")
        txt.unlink()
    parquets.append(f"'{destino}'")

br = duckdb.sql(f"""
    WITH p AS (SELECT *, UPA || '_' || V1008 || '_' || V1014 AS id_hogar FROM read_parquet([{', '.join(parquets)}])),
    hogar AS (
        SELECT Ano, Trimestre, id_hogar,
               SUM(CASE WHEN V2009 < 6 THEN 1 ELSE 0 END) AS menores_6,
               SUM(CASE WHEN V2009 < 18 THEN 1 ELSE 0 END) AS menores_18,
               SUM(CASE WHEN VD4002 = 1 THEN 1 ELSE 0 END) AS ocupados,
               MAX(CASE WHEN V2005 IN (2, 3) THEN 1 ELSE 0 END) AS hay_conyuge
        FROM p GROUP BY ALL)
    SELECT p.*, hogar.menores_6, hogar.menores_18, hogar.ocupados, hogar.hay_conyuge
    FROM p JOIN hogar USING (Ano, Trimestre, id_hogar)
    WHERE VD4002 = 1 AND VD4009 BETWEEN 1 AND 7 AND V2009 BETWEEN 18 AND 60 AND VD4016 > 0 AND V4039 BETWEEN 1 AND 98
""").df()
gran_grupo = pd.to_numeric(br["V4010"].str[:1], errors="coerce")
BR = pd.DataFrame({
    "pais": "Brasil", "periodo": (br["Ano"].astype(int) * 10 + br["Trimestre"].astype(int)).astype(str),
    "t": ((br["Ano"] - 2024) * 4 + br["Trimestre"] - 1).astype(int), "id_hogar": "BR_" + br["id_hogar"],
    "peso": br["V1028"], "mujer": (br["V2007"] == 2).astype(int), "salario_horario": br["VD4016"] / (br["V4039"] * 4.3),
    "anios_educ": br["VD3005"], "exp_pot": (br["V2009"] - br["VD3005"] - 6).clip(lower=0),
    "calificacion": gran_grupo.map(calificacion_isco).where(gran_grupo != 0, "media"),
    "rama": br["V4013"].map(lambda c: rama_comun(int(str(c).zfill(5)[:2])) if str(c).strip().isdigit() else np.nan),
    "formal": (br["VD4012"] == 1).astype(float), "temporal": (br["V4025"] == 1).astype(float),
    "jornada_parcial": (br["V4039"] < 35).astype(float),
    "en_pareja": (br["V2005"].isin([2, 3]) | ((br["V2005"] == 1) & (br["hay_conyuge"] == 1))).astype(float),
    "menores_6": br["menores_6"], "menores_18": br["menores_18"], "otros_ocupados": br["ocupados"] - 1,
    "gran_ciudad": br["V1023"].isin([1, 2]).astype(float)})
print(f"Brasil: {len(BR):,} asalariados ({BR.mujer.sum():,} mujeres) en {BR.periodo.nunique()} trimestres")

## 6. España: Encuesta de Condiciones de Vida

La ECV combina el fichero de adultos (P), el de personas (R) y el de hogares (D). El ingreso corresponde al **año
anterior** a la encuesta y las características al año de la encuesta; para que coincidan se retienen las asalariadas
que lo fueron los 12 meses del año de referencia y siguen siéndolo.

In [ ]:
def leer_ecv(anio, letra):
    with zipfile.ZipFile(RAW / "ecv" / f"datos_{anio}.zip") as z:
        interno = next(n for n in z.namelist() if re.search(rf"ECV_T{letra}_{anio}\.zip$", n, re.I))
        with zipfile.ZipFile(io.BytesIO(z.read(interno))) as z2:
            archivo = next(n for n in z2.namelist() if re.search(rf"ECV_T{letra}_{anio}\.(tab|csv)$", n, re.I))
            return pd.read_csv(io.BytesIO(z2.read(archivo)), sep="\t", low_memory=False,
                               dtype={"PE041": str, "PL051A": str, "PL111AA": str, "DB040": str})


EDUC_ES = {"000": 0, "100": 6, "200": 10, "340": 12, "344": 12, "350": 12, "353": 12, "354": 12, "450": 13, "500": 16}
partes = []
for anio in ANIOS_ECV:
    p, r, d = leer_ecv(anio, "p"), leer_ecv(anio, "r"), leer_ecv(anio, "d")
    r["hogar"] = r["RB030"] // 100
    hogar = (r.assign(menores_6=r.RB082 < 6, menores_18=r.RB082 < 18, ocupados=r.RB211 == 1)
              .groupby("hogar")[["menores_6", "menores_18", "ocupados"]].sum())
    p = (p.merge(r[["RB030", "RB082", "hogar"]], left_on="PB030", right_on="RB030")
          .join(hogar, on="hogar").merge(d[["DB030", "DB100"]], left_on="hogar", right_on="DB030"))
    p = p[(p.PL040B == 3) & (p.PL073.fillna(0) + p.PL074.fillna(0) == 12) & p.RB082.between(18, 60)
          & (p.PY010N > 0) & p.PL060.between(1, 98)]
    gran_grupo = pd.to_numeric(p["PL051A"].astype(str).str[:1], errors="coerce")
    educ = p["PE041"].astype(str).str.zfill(3).map(EDUC_ES)
    partes.append(pd.DataFrame({
        "pais": "España", "periodo": f"ECV {anio}", "t": anio - 2024, "id_hogar": "ES_" + p["hogar"].astype(str),
        "peso": p["PB040"], "mujer": (p["PB150"] == 2).astype(int), "salario_horario": p["PY010N"] / 12 / (p["PL060"] * 4.3),
        "anios_educ": educ, "exp_pot": (p["RB082"] - educ - 6).clip(lower=0),
        "calificacion": gran_grupo.map(calificacion_isco).where(gran_grupo != 0, "media"),
        "rama": p["PL111AA"].astype(str).str.upper().map(SECCION_A_RAMA),
        "formal": p["PL141"].isin([11, 21]).astype(float), "temporal": p["PL141"].isin([11, 12]).astype(float),
        "jornada_parcial": (p["PL060"] < 35).astype(float), "en_pareja": (p["PB205"] == 1).astype(float),
        "menores_6": p["menores_6"], "menores_18": p["menores_18"], "otros_ocupados": p["ocupados"] - 1,
        "gran_ciudad": (p["DB100"] == 1).astype(float)}))
ES = pd.concat(partes, ignore_index=True)
print(f"España: {len(ES):,} asalariados ({ES.mujer.sum():,} mujeres)")

## 7. Base común y variable objetivo

En cada país y período: se recortan los extremos del salario horario (0,5% superior e inferior) y se calcula
`y = log(salario horario / mediana ponderada)`.

In [ ]:
datos = pd.concat([AR, BR, ES], ignore_index=True)
limites = datos.groupby(["pais", "periodo"])["salario_horario"].quantile([0.005, 0.995]).unstack()
datos = datos.join(limites, on=["pais", "periodo"])
datos = datos[datos["salario_horario"].between(datos[0.005], datos[0.995])].drop(columns=[0.005, 0.995])
medianas = datos.groupby(["pais", "periodo"]).apply(lambda g: mediana_ponderada(g["salario_horario"], g["peso"]))
datos["y"] = np.log(datos["salario_horario"] / datos.set_index(["pais", "periodo"]).index.map(medianas))
datos["exp_pot2"] = datos["exp_pot"] ** 2
datos = datos.dropna(subset=["anios_educ", "exp_pot"]).reset_index(drop=True)
display(datos.groupby(["pais", "mujer"]).size().unstack().rename(columns={0: "varones", 1: "mujeres"}))

## 8. ¿Cuánto premia cada factor en cada país? (ecuación de Mincer común)

In [ ]:
formula = ("y ~ anios_educ + exp_pot + exp_pot2 + C(calificacion, Treatment('media')) + C(rama) + formal + temporal"
           " + jornada_parcial + en_pareja + menores_6 + menores_18 + otros_ocupados + gran_ciudad")
ETIQUETAS = {"anios_educ": "Año adicional de educación", "exp_pot": "Año de experiencia (inicial)",
             "C(calificacion, Treatment('media'))[T.alta]": "Ocupación calificada (vs. media)",
             "C(calificacion, Treatment('media'))[T.baja]": "Ocupación no calificada (vs. media)",
             "formal": "Empleo registrado", "temporal": "Contrato temporal", "jornada_parcial": "Jornada parcial",
             "en_pareja": "En pareja", "menores_6": "Cada menor de 6 en el hogar", "menores_18": "Cada menor de 18 en el hogar",
             "otros_ocupados": "Cada otro ocupado del hogar", "gran_ciudad": "Gran ciudad"}
filas = []
for pais, d in datos[datos.mujer == 1].groupby("pais"):
    d = d.dropna(subset=COMUNES + ["y"]).assign(temporal=lambda x: x.temporal.fillna(0))
    modelo = smf.wls(formula, data=d, weights=d.peso / d.peso.mean()).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(d.id_hogar)[0]})
    ic = modelo.conf_int()
    for termino, etiqueta in ETIQUETAS.items():
        if termino in modelo.params:
            filas.append({"país": pais, "variable": etiqueta, "efecto (%)": 100 * (np.exp(modelo.params[termino]) - 1),
                          "ic_inf": 100 * (np.exp(ic.loc[termino, 0]) - 1), "ic_sup": 100 * (np.exp(ic.loc[termino, 1]) - 1)})
    print(f"{pais}: n = {int(modelo.nobs):,}, R² = {modelo.rsquared:.3f}")
COEF = pd.DataFrame(filas)
display(COEF.pivot_table(index="variable", columns="país", values="efecto (%)").round(1))

fig = go.Figure()
for pais, color in zip(["Argentina", "Brasil", "España"], ["#1f77b4", "#2ca02c", "#d62728"]):
    c = COEF[COEF["país"] == pais]
    fig.add_trace(go.Scatter(x=c["efecto (%)"], y=c["variable"], mode="markers", name=pais, marker=dict(size=11, color=color),
                             error_x=dict(type="data", symmetric=False, array=c.ic_sup - c["efecto (%)"],
                                          arrayminus=c["efecto (%)"] - c.ic_inf)))
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(title="Mujeres asalariadas: efecto de cada factor sobre el salario horario (%, IC 95%)", height=600,
                  xaxis_title="Efecto sobre el salario horario (%)")
fig.show()

## 9. Capacidad predictiva y peso de cada bloque (LightGBM con las variables comunes)

En cada país se entrena con los períodos iniciales y se evalúa en el último disponible (Argentina: 1T2026; Brasil:
1T-2T2026; España: ECV 2025). La importancia SHAP se agrega en bloques.

In [ ]:
BLOQUE = {"anios_educ": "Capital humano", "exp_pot": "Capital humano", "exp_pot2": "Capital humano",
          "calificacion": "Ocupación y empresa", "rama": "Ocupación y empresa", "formal": "Condiciones de trabajo",
          "temporal": "Condiciones de trabajo", "jornada_parcial": "Condiciones de trabajo", "en_pareja": "Hogar y cuidados",
          "menores_6": "Hogar y cuidados", "menores_18": "Hogar y cuidados", "otros_ocupados": "Hogar y cuidados",
          "gran_ciudad": "Territorio"}
resultados, bloques = [], []
for pais, d in datos[datos.mujer == 1].groupby("pais"):
    X = d[COMUNES].copy()
    for c in ["calificacion", "rama"]:
        X[c] = X[c].astype("category")
    ultimo = d["t"] >= (8 if pais != "España" else 1)
    modelo = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                               random_state=SEMILLA, verbose=-1).fit(X[~ultimo], d.y[~ultimo], sample_weight=d.peso[~ultimo])
    resultados.append({"país": pais, "R² en el último período": r2_score(d.y[ultimo], modelo.predict(X[ultimo]),
                                                                         sample_weight=d.peso[ultimo])})
    shap = np.abs(modelo.predict(X[ultimo], pred_contrib=True)[:, :-1]).mean(axis=0)
    for v, valor in zip(COMUNES, shap):
        bloques.append({"país": pais, "bloque": BLOQUE[v], "shap": valor})
display(pd.DataFrame(resultados).round(3))
bloques = pd.DataFrame(bloques).groupby(["país", "bloque"])["shap"].sum().reset_index()
bloques["% de la importancia"] = 100 * bloques["shap"] / bloques.groupby("país")["shap"].transform("sum")
fig = px.bar(bloques, x="país", y="% de la importancia", color="bloque",
             title="Peso de cada bloque de determinantes en el salario de las mujeres (SHAP)")
fig.show()

## 10. Brecha de género en los tres países

Brecha bruta (diferencia de medianas del salario horario) y parte no explicada por las características, estimada con
un modelo entrenado solo con varones y aplicado a las mujeres.

In [ ]:
brechas = []
for pais, d in datos.groupby("pais"):
    X = d[COMUNES].copy()
    for c in ["calificacion", "rama"]:
        X[c] = X[c].astype("category")
    v, m = d.mujer == 0, d.mujer == 1
    modelo_v = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                                 random_state=SEMILLA, verbose=-1).fit(X[v], d.y[v], sample_weight=d.peso[v])
    brecha = np.average(d.y[v], weights=d.peso[v]) - np.average(d.y[m], weights=d.peso[m])
    no_explicada = np.average(modelo_v.predict(X[m]) - d.y[m], weights=d.peso[m])
    brechas.append({"país": pais,
                    "brecha bruta de medianas (%)": 100 * (1 - mediana_ponderada(d.salario_horario[m], d.peso[m])
                                                           / mediana_ponderada(d.salario_horario[v], d.peso[v])),
                    "brecha media (log-puntos x 100)": 100 * brecha, "no explicada": 100 * no_explicada,
                    "explicada por características": 100 * (brecha - no_explicada)})
display(pd.DataFrame(brechas).set_index("país").round(1))

## 11. Validación española con la Encuesta de Estructura Salarial (EES 2022)

La EES se basa en nóminas de empresas: mide el salario con más precisión pero no tiene variables del hogar. Si los
retornos del bloque laboral estimados con la ECV (encuesta de hogares) se parecen a los de la EES, la medición en
hogares es confiable.

In [ ]:
def leer_microdatos_zip(ruta):
    """Lee el fichero de microdatos de un zip del INE. El INE entrega los mismos datos en varios formatos (CSV, STATA,
    texto de ancho fijo "md_*.txt", etc.), a veces dentro de otro zip. Se prefiere el CSV/TAB, luego STATA (.dta); el
    texto de ancho fijo solo se usa si no hay otro, porque necesita el diseño de registro."""
    prioridad = {".csv": 0, ".tab": 0, ".dta": 1, ".txt": 2}

    def buscar(z):
        candidatos = []
        for n in z.namelist():
            extension = os.path.splitext(n.lower())[1]
            if extension == ".zip":
                candidatos += buscar(zipfile.ZipFile(io.BytesIO(z.read(n))))
            elif extension in prioridad and not re.search(r"dise|dr_|regist|leeme|readme", os.path.basename(n.lower())):
                candidatos.append((prioridad[extension], -z.getinfo(n).file_size, z, n))
        return candidatos

    candidatos = buscar(zipfile.ZipFile(ruta))
    print("Ficheros de datos en el zip:", [c[3] for c in candidatos])
    _, _, z, nombre = min(candidatos, key=lambda c: (c[0], c[1]))
    contenido = z.read(nombre)
    if nombre.lower().endswith(".dta"):
        datos = pd.read_stata(io.BytesIO(contenido), convert_categoricals=False)
        print(f"Leyendo {nombre} (STATA)")
    else:
        primera = contenido[:2000].decode("latin-1").splitlines()[0]
        sep = max(["\t", ";", ","], key=primera.count)
        print(f"Leyendo {nombre} (separador {sep!r})")
        datos = pd.read_csv(io.BytesIO(contenido), sep=sep, encoding="latin-1", low_memory=False)
    datos.columns = datos.columns.str.strip().str.upper()
    if datos.shape[1] < 5:
        raise ValueError(f"{nombre} no tiene columnas separadas (es texto de ancho fijo). Revise el contenido del zip.")
    # Códigos que deben leerse como texto (actividad, ocupación y tramo de edad)
    for col in ["CNACE", "CNO1", "ANOS2"]:
        if col in datos:
            datos[col] = datos[col].astype(str).str.strip()
    return datos


ees = leer_microdatos_zip(RAW / "ees" / "datos_2022.zip")
print(f"EES 2022: {ees.shape[0]:,} registros y {ees.shape[1]} columnas")
# Filtros: mujeres (SEXO = 6), mes completo trabajado (DRELABM = 31) y sin situaciones especiales en el mes (SIESPM1 = 6)
for nombre, condicion in [("mujeres", ees.SEXO == 6), ("mes completo", ees.DRELABM == 31),
                          ("sin situaciones especiales", ees.SIESPM1 == 6)]:
    print(f"  {nombre}: {condicion.sum():,} registros cumplen la condición")
    if condicion.sum() == 0:
        print(f"  Valores más frecuentes de la variable: {ees[condicion.name].value_counts().head(8).to_dict()}")
ees = ees[(ees.DRELABM == 31) & (ees.SIESPM1 == 6) & (ees.SEXO == 6)]
print(f"Mujeres con mes completo y sin situaciones especiales: {len(ees):,}")
horas = (ees.JSP1 + ees.JSP2 / 60) * 4.35 + ees.HEXTRA
edad = ees.ANOS2.astype(str).str.zfill(2).map({"01": 18, "02": 25, "03": 35, "04": 45, "05": 55, "06": 62})
educ = ees.ESTU.map({1: 3, 2: 6, 3: 10, 4: 12, 5: 14, 6: 15, 7: 17})
cno = ees.CNO1.astype(str).str[0]
EES = pd.DataFrame({"peso": ees.FACTOTAL, "salario_horario": (ees.SALBASE + ees.COMSAL + ees.PHEXTRA) / horas,
                    "anios_educ": educ, "exp_pot": (edad - educ - 6).clip(lower=0),
                    "calificacion": np.select([cno.isin(list("ABCD")), cno.isin(list("OP"))], ["alta", "baja"], "media"),
                    "rama": ees.CNACE.astype(str).str[0].map(SECCION_A_RAMA), "temporal": (ees.TIPOCON == 2).astype(float),
                    "jornada_parcial": (ees.TIPOJOR == 2).astype(float), "id_hogar": ees.IDENCCC.astype(str)})
EES = EES[(EES.salario_horario > 0) & edad.between(18, 60)].dropna()
print(f"EES para el modelo: {len(EES):,} asalariadas | calificación: {EES.calificacion.value_counts().to_dict()}")
EES["y"] = np.log(EES.salario_horario / mediana_ponderada(EES.salario_horario, EES.peso))
EES["exp_pot2"] = EES.exp_pot ** 2

formula_laboral = "y ~ anios_educ + exp_pot + exp_pot2 + C(calificacion, Treatment('media')) + C(rama) + temporal + jornada_parcial"
comparacion = []
for fuente, d in [("ECV 2025 (hogares)", datos[(datos.pais == "España") & (datos.mujer == 1) & (datos.t == 1)]),
                  ("EES 2022 (empresas)", EES)]:
    # Se estima solo si hay datos suficientes y las tres calificaciones (alta, media, baja)
    if len(d) < 100 or not {"alta", "media", "baja"} <= set(d.calificacion):
        print(f"{fuente}: datos insuficientes ({len(d)} registros, calificaciones {sorted(set(d.calificacion))}); se omite")
        continue
    modelo = smf.wls(formula_laboral, data=d, weights=d.peso / d.peso.mean()).fit()
    for termino in ["anios_educ", "C(calificacion, Treatment('media'))[T.alta]",
                    "C(calificacion, Treatment('media'))[T.baja]", "temporal", "jornada_parcial"]:
        comparacion.append({"fuente": fuente, "variable": ETIQUETAS[termino],
                            "efecto (%)": 100 * (np.exp(modelo.params[termino]) - 1)})
display(pd.DataFrame(comparacion).pivot_table(index="variable", columns="fuente", values="efecto (%)").round(1))

## 12. Conclusiones del capítulo comparativo

(Se completan con los resultados de la ejecución: dónde es mayor el premio a la educación y al empleo registrado, qué
peso tienen los factores del hogar en cada país y cómo se compara la parte no explicada de la brecha de género.)